# `metrics.py` — Evaluation Metrics for Multi-Aspect Sentiment

## Purpose
Provides the two evaluators used throughout training, ablation studies, and the final evaluation pipeline:

| Class | Purpose |
|-------|---------|
| `AspectSentimentEvaluator` | Per-aspect and overall classification metrics (Accuracy, Macro-F1, Weighted-F1, MCC, confusion matrix, per-class P/R/F1) |
| `MixedSentimentEvaluator` | Mixed Sentiment Resolution (MSR) metrics — how well the model handles reviews where different aspects have conflicting sentiments |

## Why Macro-F1 is the primary metric
With severe class imbalance (e.g., `price-neg` has 17 samples vs `price-pos` with 2,244), accuracy is a misleading metric — a model that always predicts `positive` achieves 98% accuracy for `price`. Macro-F1 averages F1 across all 3 classes **equally**, meaning failures on rare classes are penalised just as heavily as failures on the majority class.

In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    confusion_matrix, matthews_corrcoef, classification_report
)
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

## 1. AspectSentimentEvaluator
Computes all standard multi-class classification metrics for a single (aspect, split) combination.

In [ ]:
class AspectSentimentEvaluator:
    """
    Comprehensive evaluation for imbalanced multi-aspect sentiment classification.
    Used by experiment_runner.py to evaluate each aspect independently.
    """
    def __init__(self, aspect_names):
        self.aspect_names = aspect_names
        self.results      = {}   # Cache results by aspect name

    def evaluate_aspect(self, y_true: np.ndarray, y_pred: np.ndarray, aspect_name: str) -> dict:
        """
        Compute all metrics for a single aspect.

        Args:
            y_true:      Ground-truth integer labels (0=negative, 1=neutral, 2=positive)
            y_pred:      Predicted integer labels
            aspect_name: String name for caching (e.g. 'colour', 'overall')

        Returns:
            dict with accuracy, macro_f1, weighted_f1, mcc, confusion_matrix,
            per_class_{precision,recall,f1}, support
        """
        accuracy   = accuracy_score(y_true, y_pred)

        # Per-class P/R/F1/support for each of the 3 classes
        precision, recall, f1, support = precision_recall_fscore_support(
            y_true, y_pred, average=None, zero_division=0, labels=[0, 1, 2]
        )

        # Macro-F1: UNWEIGHTED mean across classes — the PRIMARY metric.
        # Equal weight for each class regardless of class frequency.
        _, _, macro_f1, _ = precision_recall_fscore_support(
            y_true, y_pred, average='macro', zero_division=0, labels=[0, 1, 2]
        )

        # Weighted-F1: weighted by class support — useful as secondary metric.
        _, _, weighted_f1, _ = precision_recall_fscore_support(
            y_true, y_pred, average='weighted', zero_division=0, labels=[0, 1, 2]
        )

        # MCC: single-number summary that handles imbalance well. +1 = perfect, 0 = random, -1 = inverse.
        mcc = matthews_corrcoef(y_true, y_pred)

        cm = confusion_matrix(y_true, y_pred, labels=[0, 1, 2])

        result = {
            'accuracy'           : float(accuracy),
            'macro_f1'           : float(macro_f1),
            'weighted_f1'        : float(weighted_f1),
            'mcc'                : float(mcc),
            'per_class_precision': precision.tolist(),
            'per_class_recall'   : recall.tolist(),
            'per_class_f1'       : f1.tolist(),
            'support'            : support.tolist(),
            'confusion_matrix'   : cm.tolist(),
        }
        self.results[aspect_name] = result
        return result

    def print_results(self, aspect_name: str):
        """Pretty-print all metrics for one aspect to stdout."""
        r = self.results.get(aspect_name)
        if r is None: print(f'No results for {aspect_name}'); return

        print(f"\n{'='*70}\nResults for {aspect_name.upper()}\n{'='*70}")
        print(f"Accuracy:    {r['accuracy']:.4f}")
        print(f"Macro F1:    {r['macro_f1']:.4f}")
        print(f"Weighted F1: {r['weighted_f1']:.4f}")
        print(f"MCC:         {r['mcc']:.4f}")
        print(f"\n{'Class':<12} {'Precision':<12} {'Recall':<12} {'F1':<12} {'Support'}")
        for i, cls in enumerate(['Negative', 'Neutral', 'Positive']):
            print(f"{cls:<12} {r['per_class_precision'][i]:<12.4f} {r['per_class_recall'][i]:<12.4f} {r['per_class_f1'][i]:<12.4f} {int(r['support'][i])}")

## 2. MixedSentimentEvaluator
Measures the model's ability to **simultaneously** detect when a review has conflicting sentiments across aspects.

A review is **mixed** if it has at least one aspect predicted/labelled positive AND at least one predicted/labelled negative.

The key MSR metric is **mixed_detection_rate**: out of all reviews that are genuinely mixed-sentiment, what fraction did the model correctly identify as mixed?

In [ ]:
class MixedSentimentEvaluator:
    """
    Evaluates the model's Mixed Sentiment Resolution (MSR) capability.
    Mixed sentiment = a single review has both positive AND negative aspect labels.
    """
    def __init__(self, aspect_names):
        self.aspect_names = aspect_names

    @staticmethod
    def _is_mixed(aspect_labels: dict) -> bool:
        """Returns True if the review has both positive AND negative labels across aspects."""
        vals = set(v for v in aspect_labels.values() if v is not None)
        return 0 in vals and 2 in vals  # 0=negative, 2=positive

    def evaluate_mixed_sentiment_resolution(self, review_true: dict, review_pred: dict) -> dict:
        """
        Compute MSR metrics given true and predicted labels per review.

        Args:
            review_true: {review_id: {aspect: true_label_int}}
            review_pred: {review_id: {aspect: pred_label_int}}

        Returns:
            dict with:
              mixed_review_count:    Number of truly mixed reviews in the test set
              mixed_review_accuracy: % of mixed reviews where ALL aspects are correctly predicted
              mixed_aspect_accuracy: % of individual aspect predictions within mixed reviews
              mixed_detection_rate:  % of true mixed reviews the model also predicted as mixed
        """
        mixed_review_ids      = [rid for rid, labels in review_true.items() if self._is_mixed(labels)]
        mixed_review_count    = len(mixed_review_ids)

        if mixed_review_count == 0:
            return {'mixed_review_count': 0, 'mixed_review_accuracy': 0.0,
                    'mixed_aspect_accuracy': 0.0, 'mixed_detection_rate': 0.0}

        # Per-review accuracy: ALL aspects must be correct for the review to be 'correct'
        correct_reviews    = 0
        total_aspects_seen = 0
        correct_aspects    = 0
        detected_as_mixed  = 0  # Model also predicted mixed (pos AND neg somewhere)

        for rid in mixed_review_ids:
            true_labels = review_true[rid]
            pred_labels = review_pred.get(rid, {})

            # Check if model's predictions for this review are 'mixed' too
            if self._is_mixed(pred_labels): detected_as_mixed += 1

            all_correct = True
            for asp, true_lbl in true_labels.items():
                if true_lbl is None: continue
                pred_lbl = pred_labels.get(asp)
                if pred_lbl is not None:
                    total_aspects_seen += 1
                    if pred_lbl == true_lbl: correct_aspects += 1
                    else: all_correct = False

            if all_correct: correct_reviews += 1

        return {
            'mixed_review_count'   : mixed_review_count,
            'mixed_review_accuracy': correct_reviews / mixed_review_count if mixed_review_count > 0 else 0.0,
            'mixed_aspect_accuracy': correct_aspects / total_aspects_seen if total_aspects_seen > 0 else 0.0,
            'mixed_detection_rate' : detected_as_mixed / mixed_review_count if mixed_review_count > 0 else 0.0,
        }

    def print_mixed_sentiment_results(self, metrics: dict):
        print(f"\n{'='*60}\nMixed Sentiment Resolution Results\n{'='*60}")
        print(f"Mixed reviews (test set):  {metrics['mixed_review_count']}")
        print(f"Review-level accuracy:     {metrics['mixed_review_accuracy']:.2%}")
        print(f"Aspect-level accuracy:     {metrics['mixed_aspect_accuracy']:.2%}")
        print(f"Mixed detection rate:      {metrics['mixed_detection_rate']:.2%}")